In [1]:
# Append path to deconversation modules
import sys
import os
import scanpy as sc
import numpy as np
import pandas as pd
#sys.path.append('../../deconversation')
sys.path.append("/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages")

/nfs/home/aoku/.local/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [2]:
#!pip install /gpfs/commons/groups/compbio/projects/rf_projects/dev_deconv/DECONVersation/dist/deconversation-0.0.2-py3-none-any.whl --force-reinstall --no-deps

In [3]:
## Import associated modules
# import deconversation
# from pseudobulk import *
# from preprocessing import *
# from embeddings import *
# from deconvolution import *
# from visualization import *

In [4]:
import deconversation
from deconversation import embeddings as em
from deconversation import preprocessing as pr
from deconversation import deconvolution as de
from deconversation import visualization as vs 

geneformer successfully imported.
cell2sentence is not installed. Skipping related functions.
cellhermes is not installed. Skipping related functions.
scGPT is not installed. Skipping related functions.
scVI successfully imported.


In [5]:
# Path to single-cell RNAseq data 
path = "/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/geneformer/complete/haoSub.h5ad"

In [6]:
# Read the h5ad file (just to explore columns and variables)
adata = sc.read_h5ad(path)

adata = adata[adata.obs["broad_type"].isin(
    ['B cells', 'Monocytes', 'NK cells', 'T cells', 'mDC', 'pDC']
)]

# Remove unmapped genes
#adata.var.index = adata.var.gene_name
adata = adata[:, adata.var.index.notnull()]

# Prep data for geneformer
adata = pr.load_and_prep_data(adata= adata, cell_type_col= "broad_type", mode="geneformer")

/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/deconversation/preprocessing.py:51: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["cell_id"] = adata.obs.index


In [7]:
adata.obs["broad_type"].value_counts()

broad_type
T cells      61766
Monocytes    48439
NK cells     18005
B cells      13371
mDC           2361
pDC            849
Name: count, dtype: int64

In [8]:
# Read in bulk RNAseq data
hoek_bulk = pd.read_csv("/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/bulk/hoek/hoek_counts.csv",
                       index_col=0)

# Map genes to ensembl id (required for geneformer)
hoek_bulk.index = pr.gene_id_name_map(gene_list=hoek_bulk.index, mode="to_ensembl" )

# Drop unmapped genes 
hoek_bulk = hoek_bulk.loc[hoek_bulk.index.dropna()]

In [9]:
sig_mat = pr.create_signature_matrix(adata = adata,
                                     sample_col = "batch", # sample id column
                                     cell_type_col = "broad_type",
                                     groupby = "broad_type",
                                     output_path = None)

# Transpose for embedding extraction
sig_mat = sig_mat.T

/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/deconversation/preprocessing.py:259: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  signature = expr.groupby(groupby).mean().T


In [10]:
ground_truth = pd.read_csv("/gpfs/commons/groups/compbio/projects/rf_projects/deconv_data/deconvBench/bulk/hoek/hoek_facs.csv",
                          index_col=0)

ground_truth = ground_truth.T

In [11]:
def harmonize_columns(df):
    df = df.copy()
    # Merge pDC into mDC (only if both present)
    if "pDC" in df.columns and "mDC" in df.columns:
        df["mDC"] = df["mDC"] + df["pDC"]
        df = df.drop(columns=["pDC"])
    elif "pDC" in df.columns:            # pDC alone -> treat as mDC
        df = df.rename(columns={"pDC": "mDC"})
    # Rename T cells -> T cell
    df = df.rename(columns={"T cells": "T cell"})
    return df

In [12]:
sig_mat.head()

,ENSG00000121410,ENSG00000268895,ENSG00000148584,ENSG00000175899,ENSG00000245105,ENSG00000166535,ENSG00000256661,ENSG00000128274,ENSG00000094914,ENSG00000081760,...,ENSG00000086827,ENSG00000174442,ENSG00000122952,ENSG00000198205,ENSG00000198455,ENSG00000070476,ENSG00000203995,ENSG00000162378,ENSG00000159840,ENSG00000074755
broad_type,,,,,,,,,,,,,,,,,,,,,
B cells,0.217336,0.031785,0.000449,0.001496,0.004861,0.000075,0.000075,0.000224,0.057737,0.020866,...,0.032458,0.030289,0.003814,0.025727,0.037170,0.096328,0.004637,0.127590,0.090420,0.176651
Monocytes,0.349243,0.037201,0.000062,0.002642,0.007680,0.000784,0.000186,0.000165,0.082413,0.031504,...,0.060199,0.033031,0.007411,0.010013,0.025290,0.177357,0.000145,0.337621,1.442515,0.315180
NK cells,0.086643,0.009497,0.000167,0.009497,0.040878,0.000111,0.000000,0.000222,0.059317,0.033824,...,0.036434,0.038545,0.004610,0.012996,0.022771,0.086754,0.000389,0.081811,0.359733,0.264982
T cells,0.267866,0.019023,0.000210,0.009827,0.036493,0.000097,0.000000,0.000227,0.055969,0.025775,...,0.029304,0.025435,0.013826,0.015267,0.027831,0.079575,0.000372,0.130606,0.267008,0.185847
mDC,0.357899,0.058450,0.000000,0.022025,0.024989,0.000424,0.000000,0.000000,0.122829,0.050402,...,0.111817,0.036849,0.022025,0.015671,0.037696,0.120288,0.000000,0.199068,1.050402,0.160102


In [13]:
hoek_bulk.T.head()

,ENSG00000259429,ENSG00000215529,ENSG00000231649,ENSG00000253542,ENSG00000178605,ENSG00000172771,ENSG00000121410,ENSG00000148584,ENSG00000078328,ENSG00000134864,...,ENSG00000174442,ENSG00000122952,ENSG00000198205,ENSG00000198455,ENSG00000070476,ENSG00000203995,ENSG00000162378,ENSG00000159840,ENSG00000074755,ENSG00000100181
HD30_PBMC_0,8,35,0,0,1519,16,177,1,0,170,...,224,81,128,380,3422,89,877,9428,5163,396
HD30_PBMC_1,2,35,0,0,2125,21,254,1,0,143,...,236,77,162,370,3661,84,795,13027,5945,433
HD30_PBMC_3,10,30,0,0,1630,10,247,1,0,158,...,244,97,162,441,3351,97,910,9116,5306,374
HD30_PBMC_7,18,26,0,0,1868,20,248,0,0,108,...,227,103,159,395,3511,99,784,8798,5217,346
HD31_PBMC_0,22,41,0,0,2154,18,296,1,0,148,...,312,96,168,546,4186,114,1167,13388,6362,506


## Zero-Shot 

In [14]:
layer_results = {}
metric_rows = []
celltype_rows = []

for layer in range(18, 19):
    print(f"=== layer {layer} ===", flush=True)

    sig_mat_gf_embed = em.extract_embs(
        bulk_df=sig_mat,
        mode="geneformer",
        temp_output_dir="/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/",
        model_path="ctheodoris/Geneformer",
        delete_temp_files=True,
        layer_to_quant=layer,
    )

    gf_embed = em.extract_embs(
        bulk_df=hoek_bulk.T,
        mode="geneformer",
        temp_output_dir="/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/",
        model_path="ctheodoris/Geneformer",
        delete_temp_files=True,
        layer_to_quant=layer,
    )

    results = de.run_all_deconv(bulk_df=gf_embed.T, signature_df=sig_mat_gf_embed.T)
    results = {solver: harmonize_columns(df) for solver, df in results.items()}

    layer_results[layer] = results
    
    # score each solver against ground truth
    for solver, df in results.items():
        samples = df.index.intersection(ground_truth.index)
        celltypes = df.columns.intersection(ground_truth.columns)
    
        P = df.loc[samples, celltypes].astype(float)
        T = ground_truth.loc[samples, celltypes].astype(float)
    
        # --- overall (flattened across all samples x celltypes) ---
        p = P.values.ravel()
        t = T.values.ravel()
        ok = np.isfinite(p) & np.isfinite(t)
    
        # --- per celltype ---
        for ct in celltypes:
            a = P[ct].values
            b = T[ct].values
            m = np.isfinite(a) & np.isfinite(b)
    
            # correlation is undefined if either vector is constant
            if m.sum() > 1 and np.std(a[m]) > 0 and np.std(b[m]) > 0:
                r_ct = np.corrcoef(a[m], b[m])[0, 1]
            else:
                r_ct = np.nan
    
            celltype_rows.append({
                "layer": layer,
                "solver": solver,
                "celltype": ct,
                "correlation": r_ct,
                "rmse": np.sqrt(np.mean((a[m] - b[m]) ** 2)) if m.sum() else np.nan,
            })
    
        # mean across celltypes for this layer/solver
        ct_sub = [r for r in celltype_rows if r["layer"] == layer and r["solver"] == solver]
        corrs = np.array([r["correlation"] for r in ct_sub], dtype=float)
        mean_corr = np.mean(np.nan_to_num(corrs, nan=0.0))
        mean_rmse = np.nanmean([r["rmse"] for r in ct_sub])
    
        metric_rows.append({
            "layer": layer,
            "solver": solver,
            "correlation": np.corrcoef(p[ok], t[ok])[0, 1],
            "rmse": np.sqrt(np.mean((p[ok] - t[ok]) ** 2)),
            "meanCorrelation": mean_corr,
            "meanRMSE": mean_rmse,
        })

metrics_df  = pd.DataFrame(metric_rows)
celltype_df = pd.DataFrame(celltype_rows)

#metrics_df.to_csv("../../results/gf_layer_sweep/layer_sweep_metrics_zeroshot_hoek.csv", index=False)

=== layer 18 ===
Bulk AnnData saved to: /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad
Starting Geneformer tokenization...
Tokenizing /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 21.56it/s]

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.



/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:740: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    ['B cells', 'Monocytes', 'NK cells', 'T cells', 'mDC']

    Inferred to be: categorical

  value_idx = self._prep_dim_index(value.index, attr)
/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:787: ImplicitModificationWarning: Trying to modify index of attribute `.obs` of view, initializing view as actual.
  getattr(self, attr).index = value
/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be t

Loading Geneformer model...


...

Loading tokenized dataset...
Extracting Geneformer embeddings...


  0%|          | 0/1 [00:00<?, ?it/s]

Condition number: 11.51.
~1-10: signatures are well separated; NNLS is usually hard to improve materially with another solver.
Smallest singular value: 0.2006.
If value is close to zero, the signature matrix is ill-conditioned; estimated proportions may be unstable.
Max pairwise cosine similarity between cell types: 0.9558.
Most similar pair: NK cells vs T cells.
If value is close to 1, these cell types are highly similar and difficult to resolve separately.
Running solver: nnls
Finished in 0.00 seconds.
Running solver: nnls_mod
Finished in 0.00 seconds.
Running solver: dwls
Finished in 0.01 seconds.
Running solver: simplex
Finished in 0.01 seconds.
Running solver: ridge_simplex
Finished in 0.01 seconds.
Running solver: dwls_simplex
Finished in 0.02 seconds.
Running solver: ridge
Finished in 0.01 seconds.
Running solver: elasticnet
Finished in 0.00 seconds.
Running solver: nusvr
Finished in 0.27 seconds.
Running solver: simplex_nnls
Finished in 0.01 seconds.
Running solver: gradient_de

/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/deconversation/deconvolution.py:172: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  references = torch.tensor(references, dtype=torch.float32)
/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/deconversation/deconvolution.py:174: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mixture = torch.tensor(mixture, dtype=torch.float32)


Finished in 0.41 seconds.


In [15]:
sig_mat_gf_embed.to_csv("../../../ml_deconv_data/results_bulk/gf_embeds/gf_zs_hoek_embeddings.csv")

In [16]:
# # --- visualize ---
# fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# for ax, metric in zip(axes, ["correlation", "rmse"]):
#     for solver, sub in metrics_df.groupby("solver"):
#         sub = sub.sort_values("layer")
#         ax.plot(sub["layer"], sub[metric], marker="o", markersize=4, label=solver)
#     ax.set_xlabel("Geneformer layer (layer_to_quant)")
#     ax.set_ylabel(metric)
#     ax.set_title(metric)
#     ax.set_xticks(range(1, 19))
#     ax.grid(alpha=0.3, linestyle="--")

# axes[1].legend(title="Solver", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
# fig.tight_layout()
# #fig.savefig("../../results/gf_layer_sweep/layer_sweep_zeroshot_hoek.png", dpi=300, bbox_inches="tight")
# plt.show()

# # best layer per solver
# print(metrics_df.loc[metrics_df.groupby("solver")["correlation"].idxmax()])

# # inspect any single layer with your existing plot
# #visualize_solvers(layer_results[12], ground_truth, level=["global"])

## Fine-tuned

In [17]:
layer_results = {}
metric_rows = []
celltype_rows = []

for layer in range(18, 19):
    print(f"=== layer {layer} ===", flush=True)

    sig_mat_gf_embed = em.extract_embs(
        bulk_df=sig_mat,
        mode="geneformer",
        temp_output_dir="/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/",
        model_path="/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/260624_geneformer_cellClassifier_gf_finetune/ksplit1/",
        delete_temp_files=True,
        layer_to_quant=layer,
    )

    gf_embed = em.extract_embs(
        bulk_df=hoek_bulk.T,
        mode="geneformer",
        temp_output_dir="/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/",
        model_path="/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/260624_geneformer_cellClassifier_gf_finetune/ksplit1/",
        delete_temp_files=True,
        layer_to_quant=layer,
    )

    results = de.run_all_deconv(bulk_df=gf_embed.T, signature_df=sig_mat_gf_embed.T)
    results = {solver: harmonize_columns(df) for solver, df in results.items()}

    layer_results[layer] = results
    
    # score each solver against ground truth
    for solver, df in results.items():
        samples = df.index.intersection(ground_truth.index)
        celltypes = df.columns.intersection(ground_truth.columns)
    
        P = df.loc[samples, celltypes].astype(float)
        T = ground_truth.loc[samples, celltypes].astype(float)
    
        # --- overall (flattened across all samples x celltypes) ---
        p = P.values.ravel()
        t = T.values.ravel()
        ok = np.isfinite(p) & np.isfinite(t)
    
        # --- per celltype ---
        for ct in celltypes:
            a = P[ct].values
            b = T[ct].values
            m = np.isfinite(a) & np.isfinite(b)
    
            # correlation is undefined if either vector is constant
            if m.sum() > 1 and np.std(a[m]) > 0 and np.std(b[m]) > 0:
                r_ct = np.corrcoef(a[m], b[m])[0, 1]
            else:
                r_ct = np.nan
    
            celltype_rows.append({
                "layer": layer,
                "solver": solver,
                "celltype": ct,
                "correlation": r_ct,
                "rmse": np.sqrt(np.mean((a[m] - b[m]) ** 2)) if m.sum() else np.nan,
            })
    
        # mean across celltypes for this layer/solver
        ct_sub = [r for r in celltype_rows if r["layer"] == layer and r["solver"] == solver]
        corrs = np.array([r["correlation"] for r in ct_sub], dtype=float)
        mean_corr = np.mean(np.nan_to_num(corrs, nan=0.0))
        mean_rmse = np.nanmean([r["rmse"] for r in ct_sub])
    
        metric_rows.append({
            "layer": layer,
            "solver": solver,
            "correlation": np.corrcoef(p[ok], t[ok])[0, 1],
            "rmse": np.sqrt(np.mean((p[ok] - t[ok]) ** 2)),
            "meanCorrelation": mean_corr,
            "meanRMSE": mean_rmse,
        })

metrics_df  = pd.DataFrame(metric_rows)
celltype_df = pd.DataFrame(celltype_rows)

#metrics_df.to_csv("../../results/gf_layer_sweep/layer_sweep_metrics_finetuned_hoek.csv", index=False)

=== layer 18 ===
Bulk AnnData saved to: /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad
Starting Geneformer tokenization...
Tokenizing /gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 21.15it/s]

/gpfs/commons/groups/compbio/projects/ao_projects/ml_deconv_data/DECONVersation/temp/gf_tokens.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.



/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:740: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    ['B cells', 'Monocytes', 'NK cells', 'T cells', 'mDC']

    Inferred to be: categorical

  value_idx = self._prep_dim_index(value.index, attr)
/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:787: ImplicitModificationWarning: Trying to modify index of attribute `.obs` of view, initializing view as actual.
  getattr(self, attr).index = value
/nfs/home/aoku/.local/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be t

Loading Geneformer model...


...

Loading tokenized dataset...
Extracting Geneformer embeddings...


  0%|          | 0/1 [00:00<?, ?it/s]

Condition number: 5.38.
~1-10: signatures are well separated; NNLS is usually hard to improve materially with another solver.
Smallest singular value: 0.3754.
If value is close to zero, the signature matrix is ill-conditioned; estimated proportions may be unstable.
Max pairwise cosine similarity between cell types: 0.8419.
Most similar pair: Monocytes vs mDC.
If value is close to 1, these cell types are highly similar and difficult to resolve separately.
Running solver: nnls
Finished in 0.00 seconds.
Running solver: nnls_mod
Finished in 0.00 seconds.
Running solver: dwls
Finished in 0.01 seconds.
Running solver: simplex
Finished in 0.01 seconds.
Running solver: ridge_simplex
Finished in 0.01 seconds.
Running solver: dwls_simplex
Finished in 0.02 seconds.
Running solver: ridge
Finished in 0.01 seconds.
Running solver: elasticnet
Finished in 0.00 seconds.
Running solver: nusvr
Finished in 0.27 seconds.
Running solver: simplex_nnls
Finished in 0.01 seconds.
Running solver: gradient_descen

/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/deconversation/deconvolution.py:172: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  references = torch.tensor(references, dtype=torch.float32)
/gpfs/commons/groups/compbio/projects/rf_condas/deconv_gf_test/lib/python3.10/site-packages/deconversation/deconvolution.py:174: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mixture = torch.tensor(mixture, dtype=torch.float32)


Finished in 0.40 seconds.


In [18]:
sig_mat_gf_embed.to_csv("../../../ml_deconv_data/results_bulk/gf_embeds/gf_ft_hoek_embeddings.csv")

In [30]:
#pd.DataFrame(layer_results[18]["nnls"]).to_csv("../../../ml_deconv_data/results_bulk/gf_ft_layer_18_nnls_hoek.csv")

In [28]:
# # --- visualize ---
# fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# for ax, metric in zip(axes, ["correlation", "rmse"]):
#     for solver, sub in metrics_df.groupby("solver"):
#         sub = sub.sort_values("layer")
#         ax.plot(sub["layer"], sub[metric], marker="o", markersize=4, label=solver)
#     ax.set_xlabel("Geneformer layer (layer_to_quant)")
#     ax.set_ylabel(metric)
#     ax.set_title(metric)
#     ax.set_xticks(range(1, 19))
#     ax.grid(alpha=0.3, linestyle="--")

# axes[1].legend(title="Solver", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
# fig.tight_layout()
# fig.savefig("../../results/gf_layer_sweep/layer_sweep_finetuned_hoek.png", dpi=300, bbox_inches="tight")
# plt.show()

# # best layer per solver
# print(metrics_df.loc[metrics_df.groupby("solver")["correlation"].idxmax()])

# # inspect any single layer with your existing plot
# #visualize_solvers(layer_results[12], ground_truth, level=["global"])